# P04 · LQR and model predictive control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/09_planning_learning/P04_lqr_and_model_predictive_control.ipynb)

**Track:** Planning and learning · **Skill axis (sheet):** Control and motion · **Time:** about 65 min

**Skill Hub concepts:** Linear Quadratic Regulator; Model predictive control

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("P04")

## Map: optimal control with and without constraints

Discrete system $x_{k+1} = A x_k + B u_k$. **LQR** minimizes $\sum_k x_k^\top Q x_k + u_k^\top R u_k$ forever with
$u = -Kx$, where $P$ solves the discrete algebraic Riccati equation (DARE):
$$P = Q + A^\top P A - A^\top P B\,(R + B^\top P B)^{-1} B^\top P A, \qquad K = (R + B^\top P B)^{-1} B^\top P A$$
**MPC** solves a finite-horizon version **with constraints** every sample and applies only the first input
(receding horizon). Stack the prediction $X = S_x x_0 + S_u U$ over $N$ steps, then
$$\min_U \; \lVert \bar Q^{1/2}(S_x x_0 + S_u U) \rVert^2 + \lVert \bar R^{1/2} U \rVert^2 \quad \text{s.t. } |u_k| \le u_{max}$$
which is a box-constrained least-squares problem (`scipy.optimize.lsq_linear`).

| | PID | LQR | MPC |
|---|---|---|---|
| Model needed | no | linear | linear or nonlinear |
| Constraints | clamps and anti-windup | no | yes, explicitly |
| Online cost | trivial | one matrix multiply | an optimization per sample |
| Multi-input, multi-output | awkward | natural | natural |

Plant here: double integrator (position, velocity), $\Delta t = 0.1$ s, input is acceleration.

In [ ]:
import numpy as np
from scipy.linalg import solve_discrete_are
from scipy.optimize import lsq_linear

DT = 0.1
A = np.array([[1.0, DT], [0.0, 1.0]])
B = np.array([[0.5 * DT ** 2], [DT]])

## Task 1: discrete LQR gain

In [ ]:
def dlqr(A, B, Q, R):
    """Return (K, P)."""
    raise NotImplementedError  # TODO

lab.check("lqr", dlqr)

## Task 2: condensed MPC with input bounds
`prediction(A, B, N)` returns `(Sx, Su)` with $X = [x_1; \dots; x_N] = S_x x_0 + S_u U$.
`mpc_solve(x0, A, B, Q, R, N, umax)` returns the whole optimal input sequence $U$ (length $N$, one input).

In [ ]:
def prediction(A, B, N):
    raise NotImplementedError  # TODO

def mpc_solve(x0, A, B, Q, R, N, umax):
    raise NotImplementedError  # TODO

lab.check("mpc", prediction, mpc_solve)

In [ ]:
try:
    import matplotlib.pyplot as plt
    Q, R = np.diag([10.0, 1.0]), np.array([[0.1]])
    K, _ = dlqr(A, B, Q, R)
    for name, ctrl in [("LQR, clipped", lambda x: np.clip(-K @ x, -1, 1)[0]), ("MPC, |u| <= 1", lambda x: mpc_solve(x, A, B, Q, R, 20, 1.0)[0])]:
        x, xs = np.array([2.0, 0.0]), []
        for _ in range(60):
            u = ctrl(x); x = A @ x + B[:, 0] * u; xs.append(x[0])
        plt.plot(np.arange(60) * DT, xs, label=name)
    plt.legend(); plt.xlabel("t (s)"); plt.ylabel("position"); plt.show()
except NotImplementedError:
    print("finish Tasks 1 and 2 first")

## Checkpoint: Math
Scalar system $x_{k+1} = x_k + u_k$, $Q = R = 1$.

**M1 (Apply).** Solve the scalar DARE for $P$ (positive root).

**M2 (Apply).** Closed-loop pole $1 - K$?

**M3 (Apply).** MPC horizon 20, one input: number of decision variables?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Your arm joint must never exceed its torque limit and must stay out of a keep-out zone. Which
controller handles both by design?
- a) LQR with the output clipped
- b) MPC with input and state constraints
- c) PID with a higher derivative gain

**K2 (Understand).** Explain receding horizon control and what raising $Q$ relative to $R$ does to the response.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `p04.h`).
3. In `roboprep/checks/p04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()